# アノテーション

アノテーションでは、11_record_cameraで撮影した走行データにアノテーションを実施し、転移学習をおこないます。

収集した走行データを用いて、アノテーションをし、データセットを作成します。

### ボードの識別

In [ ]:
import os

# ---------- 1. Jetson.GPIO 読み取り ----------
try:
    import Jetson.GPIO as GPIO
    BOARD_NAME = GPIO.gpio_pin_data.get_data()[0]
except Exception as e:
    # 失敗したら Orin Nano と決め打ち
    print(f"[WARN] Jetson モデル判定エラー: {e} → 強制的に JETSON_ORIN_NANO として続行")
    os.environ["JETSON_MODEL_NAME"] = "JETSON_ORIN_NANO"
    import Jetson.GPIO as GPIO          # もう一度ロード
    BOARD_NAME = "JETSON_ORIN_NANO"     # 確定

# ---------- 2. ボード別定義 ----------
mode_descriptions = {
    "JETSON_NX":       ["15W_2CORE", "15W_4CORE", "15W_6CORE", "10W_2CORE", "10W_4CORE"],
    "JETSON_XAVIER":   ["MAXN", "MODE_10W", "MODE_15W", "MODE_30W"],
    "JETSON_NANO":     ["MAXN", "5W"],
    "JETSON_ORIN":     ["MAXN", "MODE_15W", "MODE_30W", "MODE_40W"],
    "JETSON_ORIN_NANO":["MODE_15W", "MODE_25W", "MODE_MAX"]
}

product_names = {
    "JETSON_NX":        "Jetson Xavier NX",
    "JETSON_XAVIER":    "Jetson AGX Xavier",
    "JETSON_NANO":      "Jetson Nano",
    "JETSON_ORIN":      "Jetson AGX Orin",
    "JETSON_ORIN_NANO": "Jetson Orin Nano"
}

# (I2C バス番号, 初期 Power モードインデックス)
board_settings = {
    "JETSON_NX":        (8, 3),
    "JETSON_XAVIER":    (8, 2),
    "JETSON_NANO":      (1, 0),
    "JETSON_ORIN":      (7, 0),
    "JETSON_ORIN_NANO": (7, 2)
}

# ---------- 3. パラメータ取得 ----------
i2c_busnum, power_mode = board_settings.get(BOARD_NAME, (None, None))
mode_list       = mode_descriptions.get(BOARD_NAME, [])
product_name    = product_names.get(BOARD_NAME, "未知のボード")

# ---------- 4. 出力 ----------
if i2c_busnum is not None and 0 <= power_mode < len(mode_list):
    mode_str = mode_list[power_mode]
    print("------------------------------------------------------------")
    print(f"{product_name} を認識: I2C バス番号 = {i2c_busnum}, "
          f"Power モード = {mode_str} ({power_mode})")
    print("------------------------------------------------------------")
else:
    raise RuntimeError(f"未対応の Jetson モデル、または Power モード定義不足: {BOARD_NAME}")

In [ ]:
!echo "jetson" | sudo -S nvpmodel -m $power_mode

In [ ]:
!echo "jetson" | sudo -S nvpmodel -q

In [ ]:
!echo "jetson" | sudo -S jetson_clocks

走行データはcameraフォルダに録画されています。今度は、cameraフォルダのデータにアノテーションをおこないます。

In [ ]:
import os
import re
import threading
import time
from datetime import datetime, timezone

import cv2
import ipywidgets
import matplotlib.image as mpimg
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
import re
import torch
import torchvision
import torchvision.transforms as transforms
from ipywidgets import Button, Layout, Textarea, HBox, VBox, Label
from jetcam.utils import bgr8_to_jpeg
from jupyter_clickable_image_widget import ClickableImageWidget

from fabo import asset_root
from fabo.annotation import draw_grids
from utils import preprocess
from xy_dataset import XYDataset

In [ ]:
IMG_WIDTH = 224
IMG_HEIGHT = 224

SLEEP = [50,100,200,300,400,500]
SKIP = [1,2,3,4,5]

LOAD_CATEGORIES = ['xy','speed']
SAVE_CATEGORIES = ['xy','speed']

LOAD_TASK = ['camera','dataset','interactive']
SAVE_TASK = ['dataset']

check_flag = False
running = False
sleep_time = 30

def get_dirs(path):
    files = os.listdir(path)
    dirs = [f for f in files if os.path.isdir(os.path.join(path, f))]
    dirs = [f for f in dirs if f != ".ipynb_checkpoints"]
    dirs = sorted(dirs)

    return dirs

TRANSFORMS = transforms.Compose([
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

datasets = {}


In [ ]:
l = Layout(flex='0 1 auto', height='100px', min_height='100px', width='auto')
process_widget = ipywidgets.Textarea(description='ログ', value='', layout=l)

process_no = 0
def write_log(msg):
    global process_widget, process_no
    process_no = process_no + 1
    process_widget.value = str(process_no) + ": " + msg + "\n" + process_widget.value

In [ ]:
sleep_dropdown = ipywidgets.Dropdown(options=SLEEP, description='sleep(ms)', index=1)
skip_dropdown = ipywidgets.Dropdown(options=SKIP, description='skip(枚)', index=1)
skip_movie_dropdown = ipywidgets.Dropdown(options=SKIP, description='skip(枚)', index=1)

picture_widget = ClickableImageWidget(width=224, height=224)
picture_widget.format = "jpeg"

no_widget = ipywidgets.IntText(description='no')
x_widget = ipywidgets.IntText(description='data x')
y_widget = ipywidgets.IntText(description='data y')
speed_widget = ipywidgets.IntText(description='data speed')
ai_x_widget = ipywidgets.IntText(description='AI　x')
ai_y_widget = ipywidgets.IntText(description='AI　y')
ai_speed_widget = ipywidgets.IntText(description='AI speed')
model_widget = ipywidgets.Text(description='model')
model_widget.value = "model.pth"
load_model_button = ipywidgets.Button(description='load model')
speed_slider = ipywidgets.IntSlider(description='speed', min=0, max=224, step=1, value=0, orientation='vertical')
add_speed_button = ipywidgets.Button(description='速度追加')

In [ ]:
from packaging import version

torchvision_version = version.parse(torchvision.__version__)

device = torch.device('cuda')
output_dim = 2 * len(LOAD_CATEGORIES)
model = None
model_metadata = {}

In [ ]:
from pathlib import Path

import yaml

from fabo import asset_root


def load_model(c):
    global model, model_metadata

    def new_model_metadata():
        return {
            "model": None,
            "model_type": "ResNet18",
            "torchvision_version": str(torchvision_version),
        }

    def load_model_metadata(_model_name):
        p = Path(_model_name)
        load_model_metadata_path = p.parent / (p.stem + ".yaml")
        if load_model_metadata_path.exists():
            try:
                with open(load_model_metadata_path, "w") as f:
                    _model_metadata = yaml.safe_load(f)
            except Exception as e:
                write_log(f"{load_model_metadata_path}のモデルメタデータ読込に失敗: {e}")
                _model_metadata = {}
        else:
            _model_metadata = {}
        _model_metadata |= {
            "model": str(Path(_model_name).relative_to(asset_root())),
            "model_type": "ResNet18",
            "torchvision_version": str(torchvision_version),
        }
        return _model_metadata

    model_name = load_model_widget.value
    if torchvision_version >= version.parse("0.13"):
        # torchvision 0.13以降の場合
        from torchvision.models.resnet import ResNet18_Weights, resnet18
        
        if model_name == "[new]":
            # 新しい重みを使ってモデルをロード
            default_weights = ResNet18_Weights.DEFAULT
            model = resnet18(weights=default_weights)
            model.fc = torch.nn.Linear(model.fc.in_features, output_dim)
            model_metadata = new_model_metadata()
            write_log("[new]が選択されたのでresnet18の最新の重みから始めます(torchvision 0.13以降)。")
        else:
            model = resnet18(weights=None)  # pretrained=Falseの代わり
            model.fc = torch.nn.Linear(model.fc.in_features, output_dim)
            state_dict = torch.load(model_name, weights_only=True)
            model.load_state_dict(state_dict)
            model_metadata = load_model_metadata(model_name)
            write_log(model_name + "のモデルを読込ました(torchvision 0.13以降)。")

    else:
        # torchvision 0.13より前の場合
        if model_name == "[new]":
            model = torchvision.models.resnet18(pretrained=True)
            model.fc = torch.nn.Linear(512, output_dim)
            write_log("[new]が選択されたのでresnet18のpretrainedから始めます。")
        else:
            model = torchvision.models.resnet18(pretrained=False)
            model.fc = torch.nn.Linear(512, output_dim)
            model.load_state_dict(torch.load(model_name))
            model_metadata = load_model_metadata(model_name)
            write_log(model_name + "のモデルを読込ました。")

    model.eval()
    model = model.to(device)
    get_jetson_nano_memory_usage()


def save_model(c):
    global model_metadata
    path = os.path.join(asset_root(), "model")
    os.makedirs(path, exist_ok=True)
    save_model_path = os.path.join(path, save_model_name_widget.value)
    if save_best_model_checkbox.value:
        import shutil
        save_model_best_path = os.path.join(path, "best_model.pth")
        shutil.move(save_model_best_path, save_model_path)
    else:
        torch.save(model.state_dict(), save_model_path)

    p = Path(save_model_path)
    model_metadata |= {
        "model": str(p.relative_to(asset_root())),
        "save_best_model": save_best_model_checkbox.value,
    }
    save_model_metadata_path = p.parent / (p.stem + ".yaml")
    with open(save_model_metadata_path, "w") as f:
        yaml.dump(model_metadata, f)

    write_log(save_model_path + "に保存しました。")


In [ ]:
import time

from fabo import asset_root, format_epoch

BATCH_SIZE = 8

epochs_widget = ipywidgets.IntText(description='epochs', value=1)
eval_button = ipywidgets.Button(description='evaluate')
train_button = ipywidgets.Button(description='train')
loss_widget = ipywidgets.FloatText(description='loss')
best_loss_form = widgets.Text(description='best_loss', disabled=True, value="inf")
reset_best_loss_button = ipywidgets.Checkbox(description="reset best_loss", value=False)
progress_widget = ipywidgets.FloatProgress(min=0.0, max=1.0, description='progress')

best_loss = float('inf')

def train_eval(is_training):
    global best_loss, model, model_metadata
    
    optimizer = torch.optim.Adam(model.parameters())

    save_dataset_name = save_datasets_widget.value
    xy_path = os.path.join(asset_root(), save_task_widget.value, save_dataset_name, "xy")
    speed_path = os.path.join(asset_root(), save_task_widget.value, save_dataset_name, "speed")

    xy_is_dir = os.path.isdir(xy_path)
    speed_is_dir = os.path.isdir(speed_path)
    
    xy_file_count = 0
    speed_file_count = 0
    
    if xy_is_dir:
        xy_file_count = sum(os.path.isfile(os.path.join(xy_path,name)) for name in os.listdir(xy_path))
    if speed_is_dir:
        speed_file_count = sum(os.path.isfile(os.path.join(speed_path,name)) for name in os.listdir(speed_path))

    save_dataset_path = os.path.join(asset_root(), save_task_widget.value, save_dataset_name)

    # ベースモデル情報を学習前モデル情報に設定し、モデル情報を初期化する
    model_metadata |= {
        "base_model": model_metadata["model"],
        "model": None,
    }
    # 各種学習構成を設定
    model_metadata |= {
        "batch_size": BATCH_SIZE,
        "dataset": str(Path(save_dataset_path).relative_to(asset_root())),
        "dataset_categories": {
            "xy": {"samples_size": xy_file_count},
            "speed": {"samples_size": speed_file_count},
        },
        "epochs": epochs_widget.value,
        "sample_size": xy_file_count + speed_file_count,
    }

    write_log("-------------------------")
    write_log("学習を開始します。")
    write_log("データセット: " + save_dataset_path)
    write_log("XYデータ数: " + str(xy_file_count) + " Speedデータ数: " + str(speed_file_count))
    write_log("-------------------------")    

    dataset = XYDataset(save_dataset_path, SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)
    train_button.disabled = True
    eval_button.disabled = True
        
    # 有効なデータのインデックスを確認
    valid_indices = []
    for i in range(len(dataset)):
        try:
            item = dataset[i]
            if item is not None and item[0] is not None:
                valid_indices.append(i)
        except Exception as e:
            pass
        
    # 有効なインデックスを持つサブセットを作成
    valid_dataset = torch.utils.data.Subset(dataset, valid_indices)

    try:
        train_loader = torch.utils.data.DataLoader(
            valid_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True
        )
        time.sleep(1)

        if is_training:
            model = model.train()
        else:
            model = model.eval()
        epoch_count = 0
        global_start_time = time.time()

        while epochs_widget.value > 0:
            epoch_start_time = time.time()  # エポック開始時間を記録
            epoch_count += 1
            i = 0
            sum_loss = 0.0
            error_count = 0.0
            for images, category_idx, xy in iter(train_loader):
                if images is None or xy is None:
                    print("Warning: None type data found at index", i)
                    continue
                images = images.to(device)
                xy = xy.to(device)

                if is_training:
                    # zero gradients of parameters
                    optimizer.zero_grad()

                # execute model to get outputs
                outputs = model(images)

                # compute MSE loss over x, y coordinates for associated categories
                loss = 0.0
                for batch_idx, cat_idx in enumerate(list(category_idx.flatten())):
                    loss += torch.mean((outputs[batch_idx][2 * cat_idx:2 * cat_idx+2] - xy[batch_idx])**2)
                loss /= len(category_idx)

                if is_training:
                    # run backpropogation to accumulate gradients
                    loss.backward()

                    # step optimizer to adjust parameters
                    optimizer.step()

                # increment progress
                count = len(category_idx.flatten())
                i += count
                sum_loss += float(loss.item())
                progress_widget.value = i / len(dataset)
                loss_widget.value = sum_loss / i
            
            # エポック終了時に時間を記録してログに出力
            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - epoch_start_time
            write_log(f"{epoch_count} Epoch目: {epoch_duration:.2f}秒")
            #get_jetson_nano_memory_usage()
            
            # 最小損失をチェックし、必要に応じてモデルを保存
            if is_training and loss_widget.value < best_loss:
                best_loss = loss_widget.value
                best_loss_form.value = f"{best_loss:.7f}"
                model_dir = os.path.join(asset_root(), 'model')
                if not os.path.exists(model_dir):
                    os.makedirs(model_dir)
                torch.save(model.state_dict(), os.path.join(model_dir, 'best_model.pth'))
                write_log(f"新しいベストモデルが保存されました。Epoch loss: {best_loss:.4f}")

            if is_training:
                epochs_widget.value = epochs_widget.value - 1
            else:
                break

        global_end_time = time.time()
        global_duration = global_end_time - global_start_time
        model_metadata |= {
            "duration": global_duration,
            "end_time": format_epoch(global_end_time),
            "metrics": {
                "best_loss": best_loss,
                "final_loss": loss_widget.value,
            },
            "start_time": format_epoch(global_start_time),
        }
    except Exception as e:
        pass

    model = model.eval()

    train_button.disabled = False
    eval_button.disabled = False

train_button.on_click(lambda c: train_eval(is_training=True))
eval_button.on_click(lambda c: train_eval(is_training=False))

def reset_best_loss(change):
    global best_loss
    best_loss = float("inf")
    best_loss_form.value = "inf"
reset_best_loss_button.on_click(reset_best_loss)

train_eval_widget = ipywidgets.VBox([
    epochs_widget,
    progress_widget,
    loss_widget,
    ipywidgets.HBox([best_loss_form, reset_best_loss_button]),
    ipywidgets.HBox([train_button, eval_button]),
])

display(train_eval_widget)

01_find_pwmを実行して、pwmの値を設定してください。

In [ ]:
import Fabo_PCA9685
import time
import smbus
import time
import json

with open('pwm_params.json') as f:
    json_str = json.load(f)

    stop = json_str["pwm_speed"]["stop"]
    left = json_str["pwm_steering"]["left"]
    center = json_str["pwm_steering"]["center"]
    right = json_str["pwm_steering"]["right"]

if stop == 0:
    INITIAL_VALUE=400
else:
    INITIAL_VALUE=stop

In [ ]:
def map_rc(x, in_min, in_max, out_min, out_max):
    return (x - in_min) * (out_max - out_min) // (in_max - in_min) + out_min

def handle(x):
    x = map_rc(x, 224, 0, right, left)

In [ ]:
import functools
import glob
import subprocess
from os.path import join

from fabo import asset_root

def extract_numbers(filename):
    matches = re.findall(r'(\d+)', filename)
    if matches and len(matches) >= 3: 
        return int(matches[-1])  
    else:
        return float('inf') 

def get_file_names(path):
    file_names = os.listdir(path)
    file_names = [os.path.join(path, file_name) for file_name in file_names]
    image_names = []

    image_names = sorted(file_names, key=lambda f: extract_numbers(os.path.basename(f)))
    image_names = [f for f in image_names if os.path.splitext(f)[1].lower() == ".jpg"]
    
    return image_names

@functools.lru_cache(maxsize=1024)
def load_img_from_disk(path):
    return cv2.imread(path, cv2.IMREAD_COLOR) 

def load_img(no):
    global img, img_filename, load_flag, play_num, running, xy_filenames
    load_task_value = load_task_widget.value
    datasets_value = load_datasets_widget.value

    xy_path = os.path.join(asset_root(), load_task_value, datasets_value, "xy")
    speed_path = os.path.join(asset_root(), load_task_value, datasets_value, "speed")
    
    t_all = time.perf_counter()
    
    xy_imagenames = get_file_names(xy_path)
            
    if no >= len(xy_imagenames):
        no_widget.value = no - 1
        write_log("ファイルが存在しません。" + str(len(xy_imagenames)-1) + "以内の値を設定してください。")
        running = False
        return
        
    xy_name = xy_imagenames[no]
    
    pattern = '.*/(\d+)_(\d+)_.*'
    xy_result = re.match(pattern, xy_name)
    if xy_result:
        x = xy_result.group(1)
        y = xy_result.group(2)
        x_widget.value = x
        y_widget.value = y

    speed = 0
    try:
        speed_imagenames = get_file_names(speed_path)
        speed_name = speed_imagenames[no]
        speed_result = re.match(pattern, speed_name)
        if speed_result:
            speed = int(speed_result.group(2))
            speed_widget.value = speed
            speed_slider.value = speed
    except:
        speed_widget.value = 0
        
    t0 = time.perf_counter()
    #img = cv2.imread(xy_name)
    img = load_img_from_disk(xy_name)
    load_ms = (time.perf_counter() - t0) * 1000
    
    if img is None:
        write_log("Image could not be loaded: " + xy_name)
        return
    img_filename = xy_name

    marked_img = img.copy()
    
    black_color = (0, 0, 0)
    blue_color = (255, 0, 0)
    green_color = (0, 255, 0)
    
    if int(x) != 0 or int(y) != 0:
        marked_img = cv2.circle(marked_img, (int(x), int(y)), 8, green_color, 3)
    
    try:
        t0 = time.perf_counter()
        preprocessed = preprocess(img)
        output = model(preprocessed).detach().cpu().numpy().flatten()
        infer_ms = (time.perf_counter() - t0) * 1000
        
        t0 = time.perf_counter()
        result_x = output[0]
        result_y = output[1]
        result_speed = output[3]
        result_x = int(IMG_WIDTH * (result_x / 2.0 + 0.5))
        result_y = int(IMG_HEIGHT * (result_y / 2.0 + 0.5))
        result_speed = int(IMG_HEIGHT * (result_speed / 2.0 + 0.5))
        
        handle(result_x)
        ai_x_widget.value = result_x
        ai_y_widget.value = result_y
        ai_speed_widget.value = result_speed
        
        marked_img = cv2.circle(marked_img, (int(result_x), int(result_y)), 8, blue_color, 3)
        marked_img = draw_grids(marked_img)
        
        # Speed
        if result_speed> 224:
            result_speed = 224
        elif result_speed < 0:
            result_speed = 0
            
        marked_img = cv2.line(marked_img,(218,0),(218,224),black_color,5)
        marked_img = cv2.line(marked_img,(219,224-result_speed),(219,224),blue_color,3)
        marked_img = cv2.putText(marked_img,"speed:"+str(result_speed),(160,215),cv2.FONT_HERSHEY_SIMPLEX,0.3,(255,255,255))
    
        if int(speed) != 0:
            marked_img = cv2.line(marked_img,(1,0),(1,224),black_color,5)
            marked_img = cv2.line(marked_img,(2,224-int(speed)),(2,224),green_color,3)        
    except Exception as e:
        write_log(f"エラーが発生しました: {e}")
    
    draw_ms = (time.perf_counter() - t0) * 1000
    
    t0 = time.perf_counter()
    picture_widget.value = bgr8_to_jpeg(marked_img)
    enc_ms = (time.perf_counter() - t0) * 1000
    
    total_ms = (time.perf_counter() - t_all) * 1000
    write_log(
        f"{no} 枚目 {os.path.basename(xy_name)} "
        f"load:{load_ms:.1f}ms infer:{infer_ms:.1f}ms "
        f"draw:{draw_ms:.1f}ms enc:{enc_ms:.1f}ms total:{total_ms:.1f}ms"
    )

    if running == True:
        play_num += 1
        if play_num % 10 == 0:
            write_log(f"{play_num} 回目の再生")
    else:
        next_image_button.disabled = False
        prev_image_button.disabled = False
        write_log(str(no) + "枚目の" + xy_name + "を読込ました。") 

def del_pic(c):
    global no, load_task_widget,load_datasets_widget, xy_filenames
    no = no_widget.value
    name = xy_filenames[no]
    os.remove(name)
    write_log(name + "を削除しました。")
    
def load_dataset(c):
    global img, load_flag
    dataset_path = os.path.join(asset_root(), load_task_widget.value, load_datasets_widget.value)
    write_log("データセット: " + dataset_path + "を読込みます。")
    load_flag = True
    no = 0
    write_log("初回の読込みには時間がかかります。(30秒〜1分)")
    load_img(no)
    get_jetson_nano_memory_usage()

def next_pic(c):
    global no,x,y,load_flag,skip,check_flag
    load_flag = True
    check_flag = False
    no = no_widget.value
    no = int(no) + skip_dropdown.value
    no_widget.value = no
    next_image_button.disabled = True
    prev_image_button.disabled = True
    load_img(no)
    
def before_pic(c):
    global no,x,y,load_flag,skip,check_flag
    load_flag = True
    check_flag = False
    no = no_widget.value
    no = int(no) - skip_dropdown.value
    if no < 0:
        no = 0
    no_widget.value = no
    next_image_button.disabled = True
    prev_image_button.disabled = True
    load_img(no)

def file_count():
    try:
        xy_path = os.path.join(asset_root(), save_task_widget.value, save_datasets_widget.value, "xy")
        speed_path = os.path.join(asset_root(), save_task_widget.value, save_datasets_widget.value, "speed")
        
        xy_is_dir = os.path.isdir(xy_path)
        
        if xy_is_dir:
            xy_file_count = sum(os.path.isfile(os.path.join(xy_path,name)) for name in os.listdir(xy_path))
            datasets_xy_count_widget.value = xy_file_count
        else:
            datasets_xy_count_widget.value = 0
        
        speed_is_dir = os.path.isdir(speed_path)
        
        if speed_is_dir:
            speed_file_count = sum(os.path.isfile(os.path.join(speed_path,name)) for name in os.listdir(speed_path))
            datasets_speed_count_widget.value = speed_file_count
        else:
            datasets_speed_count_widget.value = 0
            
    except Exception as e:
        #print("An error occurred:", e)
        datasets_xy_count_widget.value = 0
        datasets_speed_count_widget.value = 0


In [ ]:
import datetime
import shutil
from pathlib import Path

from fabo import asset_root

def append_to_parquet(dataset_path, saved_path, input_image_path, value_type, value):
    """
    Parquetファイルに行を追加する、または既存レコードを更新する

    Args:
        dataset_path: データセットのパス (例: "dataset/dataset1")
        saved_path: 出力先パス
        input_image_path: 入力画像パス
        value_type: 値種別 ("x", "y", "speed")
        value: 値
    """
    parquet_path = os.path.join(dataset_path, "dataset_v1.parquet")
    input_image_path = str(Path(input_image_path).relative_to(asset_root()))

    # ファイル名から InX, InY, InFrameNum をパース
    stem = Path(input_image_path).stem
    in_x, in_y, in_frame_num = None, None, None
    try:
        parts = stem.split('_')
        if len(parts) == 3:
            in_x = int(parts[0])
            in_y = int(parts[1])
            in_frame_num = int(parts[2])
    except (ValueError, IndexError):
        pass  # パースに失敗した場合は None のまま

    # 既存のParquetファイルがあれば読み込み
    if os.path.exists(parquet_path):
        existing_df = pd.read_parquet(parquet_path)

        # 同じ入力画像パス、値種別のレコードを検索
        existing_records = existing_df[
            (existing_df['入力画像パス'] == input_image_path)
            & (existing_df['値種別'] == value_type)
        ]

        if not existing_records.empty:
            # 複数のレコードが存在する場合は警告
            if len(existing_records) > 1:
                write_log(f"⚠️ 警告: 同じ入力画像に対して複数のアノテーションが存在します（{len(existing_records)}件）")
                write_log(f"⚠️ データセットの整合性に問題があります。dataset_v1.parquet を確認し、重複したレコードと画像ファイルを削除してください。")
                write_log(f"⚠️ 入力画像パス: {input_image_path}")

            # 既存レコードが存在する場合
            existing_record = existing_records.iloc[0]
            old_value = existing_record['値']
            old_saved_path = os.path.join(asset_root(), existing_record['出力先パス'])

            if old_value != value:
                # 値が異なる場合、ファイルを移動
                if os.path.exists(old_saved_path):
                    if os.path.exists(saved_path):
                        # 移動先に既にファイルが存在する場合は削除
                        os.remove(saved_path)

                    # 移動先ディレクトリが存在しない場合は作成
                    saved_dir = os.path.dirname(saved_path)
                    if not os.path.exists(saved_dir):
                        os.makedirs(saved_dir)

                    # ファイルを移動
                    shutil.move(old_saved_path, saved_path)

                    # ファイルの変更時刻を更新（現在時刻に設定）
                    current_time = datetime.datetime.now().timestamp()
                    os.utime(saved_path, (current_time, current_time))

                    write_log(f"ファイルを移動: {old_saved_path} -> {saved_path}")

                # Parquetの該当行を更新
                existing_df.loc[existing_records.index[0], '出力先パス'] = saved_path
                existing_df.loc[existing_records.index[0], '値種別'] = value_type
                existing_df.loc[existing_records.index[0], '値'] = value
                existing_df.loc[existing_records.index[0], 'UTCタイムスタンプ'] = datetime.datetime.now(timezone.utc).isoformat()

                # Parquetファイルに保存
                existing_df.to_parquet(parquet_path, index=False)
                write_log(f"既存アノテーションを更新: {value_type}={old_value} -> {value_type}={value}")
                return
            else:
                # 同じ場合は何もしない
                if os.path.exists(saved_path):
                    # 既にファイルが存在する場合は削除
                    os.remove(saved_path)
                write_log(f"既に同じ値でアノテーション済み: {value_type}={value}")
                return
    else:
        existing_df = None

    # 新しい行のデータ
    new_data = {
        '出力先パス': [saved_path],
        '入力画像パス': [input_image_path],
        'InX': [in_x],
        'InY': [in_y],
        'InFrameNum': [in_frame_num],
        '値種別': [value_type],
        '値': [value],
        'UTCタイムスタンプ': [datetime.datetime.now(timezone.utc).isoformat()]
    }
    new_df = pd.DataFrame(new_data)

    # 既存のParquetファイルがあれば追加、なければ新規作成
    if existing_df is not None:
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined_df = new_df

    # Parquetファイルに保存
    combined_df.to_parquet(parquet_path, index=False)

def save_snapshot(_, content, msg):
    global img,x,y,load_flag,save_datasets_widget,save_task_widget
    if content['event'] == 'click' and load_flag == True:
        load_flag = False
        data = content['eventData']
        x = min(max(data['offsetX'], 0), IMG_WIDTH)  # 0 <= x <= IMG_WIDTH
        # y = data['offsetY']
        y = 112  # y の利用が非推奨、かつ可視化の都合、最小・最大の0・224の画像端より中央が見やすいのでニュートラル値に設定

        remarked_img = img.copy()
        remarked_img = cv2.circle(remarked_img, (int(x), int(y)), 8, (0, 255, 0), 3)
        picture_widget.value = bgr8_to_jpeg(remarked_img)
        name = save_datasets_widget.value
        if save_task_widget.value == "":
            write_log("データセット名を指定してください")
        else:
            write_log("["+save_task_widget.value + "/" + name + "]の" + SAVE_CATEGORIES[0] + "カテゴリにデータを追加しました。")
            _dataset = datasets[name]
            saved_path = _dataset.save_entry("xy", img_filename, x, y)

            # Parquetにx, y の2行を追加
            dataset_path = os.path.join(asset_root(), save_task_widget.value, name)
            append_to_parquet(dataset_path, saved_path, img_filename, "x", x)
            append_to_parquet(dataset_path, saved_path, img_filename, "y", y)

            file_count()

def save_speed(c):
    global img,speed_slider,save_datasets_widget,save_task_widget
    speed = speed_slider.value
    remarked_img = img.copy()
    name = save_datasets_widget.value
    _dataset = datasets[name]
    saved_path = _dataset.save_entry("speed", img_filename, 0, speed)
    write_log("["+save_task_widget.value + "/" + name + "]の" + SAVE_CATEGORIES[1] + "カテゴリにデータを追加しました。")

    # Parquetにspeedの1行を追加
    dataset_path = os.path.join(asset_root(), save_task_widget.value, name)
    append_to_parquet(dataset_path, saved_path, img_filename, "speed", speed)

    file_count()


In [ ]:
from fabo import asset_root

def live():
    global no,running, skip, sleep_time, play_num
    load_flag = True
    play_num = 0
    no = no_widget.value
    while running:
        no += skip
        no_widget.value = no
        try:
            load_img(no)
        except:
            write_log("no: " + no + "のファイルの読込に失敗")
        time.sleep(sleep_time/1000)

def play(c):
    global running, execute_thread, skip, sleep_time, check_flag
    skip = skip_dropdown.value
    sleep_time = sleep_dropdown.value
    running = True
    check_flag = False
    execute_thread = threading.Thread(target=live)
    execute_thread.start()

def stop(c):
    global running, execute_thread, load_flag, check_flag
    running = False
    load_flag = True
    check_flag = False
    try:
        execute_thread.join()
        write_log("STOP")
    except:
        write_log("現在再生されていません。")

def create_dataset(c):
    new_dataset_name = datasets_name_widget.value

    # データセットのディレクトリおよびオブジェクトの作成
    path = os.path.join(asset_root(), save_task_widget.value, new_dataset_name)
    os.makedirs(path)
    datasets[new_dataset_name] = XYDataset(path, SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)

    # データセット一覧のウィジェットの更新
    change_save_task(None)
    save_datasets_widget.value = new_dataset_name

    write_log("Datasetを作成しました：" + new_dataset_name)


picture_widget.on_msg(save_snapshot)

# 画像の操作
play_button = ipywidgets.Button(description='▶')
stop_button = ipywidgets.Button(description='⏹')
next_image_button = ipywidgets.Button(description='>')
prev_image_button = ipywidgets.Button(description='<')
load_image_button = ipywidgets.Button(description='読込')
update_image_button = ipywidgets.Button(description='更新')
delete_image_button = ipywidgets.Button(description='削除')

save_model_button = ipywidgets.Button(description='save model')
save_best_model_checkbox = ipywidgets.Checkbox(description='Bestモデルを保存', value=True)

load_model_widget = ipywidgets.Dropdown(options=[],description='読込モデル')
load_model_refresh_button = ipywidgets.Button(description="↻", button_style="info", layout=ipywidgets.Layout(width="30px"))
load_model_time_widget = ipywidgets.Text(description='作成日時')
load_datasets_refresh_button = ipywidgets.Button(description="↻", button_style="info", layout=ipywidgets.Layout(width="30px"))
save_model_name_widget = ipywidgets.Text(description='保存モデル名',value="model.pth")
save_datasets_refresh_button = ipywidgets.Button(description="↻", button_style="info", layout=ipywidgets.Layout(width="30px"))

dataset_create_button = ipywidgets.Button(description='Create dataset')
datasets_name_widget = ipywidgets.Text(description='Name')

dataset_create_button.on_click(create_dataset)

play_button.on_click(play)
stop_button.on_click(stop)

add_speed_button.on_click(save_speed)

load_model_button.on_click(load_model)
save_model_button.on_click(save_model)
load_image_button.on_click(load_dataset)
next_image_button.on_click(next_pic)
prev_image_button.on_click(before_pic)
delete_image_button.on_click(del_pic)

load_datasets_widget = ipywidgets.Dropdown(options=[], description='dataset')
save_datasets_widget = ipywidgets.Dropdown(options=[], description='dataset')
datasets_xy_count_widget = ipywidgets.IntText(description='XYデータ数')
datasets_speed_count_widget = ipywidgets.IntText(description='速度データ数')

def set_dataset(change):
    datasets[change['new']] = XYDataset(
        os.path.join(asset_root(), save_task_widget.value, change['new']),
        SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)
save_datasets_widget.observe(set_dataset, names='value')

load_task_widget = ipywidgets.Dropdown(options=LOAD_TASK, description='task')
save_task_widget = ipywidgets.Dropdown(options=SAVE_TASK,  value=SAVE_TASK[0], description='task')

def change_load_task(change):
    path = os.path.join(asset_root(), load_task_widget.value)
    try:
        dirs = get_dirs(path)
        load_datasets_widget.options = dirs
        if len(load_datasets_widget.options) > 0 and load_datasets_widget.value not in load_datasets_widget.options:
            load_datasets_widget.index = 0
    except:
        write_log(path + "が存在していません。")
        load_datasets_widget.options = []
load_datasets_refresh_button.on_click(change_load_task)
load_task_widget.observe(change_load_task, names='value')
change_load_task(None)

def change_save_task(change):
    path = os.path.join(asset_root(), save_task_widget.value)
    try:
        os.makedirs(path, exist_ok=True)
        dirs = get_dirs(path)
        save_datasets_widget.options = dirs
        # 保存先データセットは明示的に指定させる
        # if len(save_datasets_widget.options) > 0 and save_datasets_widget.value not in save_datasets_widget.options:
        #     save_datasets_widget.index = 0
    except:
        write_log(path + "が存在していません。")
        save_datasets_widget.options = ['']
save_datasets_refresh_button.on_click(change_save_task)
save_task_widget.observe(change_save_task, names='value')
change_save_task(None)

def change_save_dataset(change):
    file_count()
save_datasets_widget.observe(change_save_dataset, names='value')
change_save_dataset(None)

def change_sleep(change):
    global sleep_time
    sleep_time = sleep_dropdown.value
sleep_dropdown.observe(change_sleep, names='value')

def change_skip(change):
    skip = skip_dropdown.value
skip_dropdown.observe(change_skip, names='value')

def model_list(change):
    try:
        files = glob.glob(os.path.join(asset_root(), 'model', '*.pth'), recursive=True)
        files.insert(0,"[new]")
        load_model_widget.options = files
        if len(load_model_widget.options) > 0 and load_model_widget.value not in load_model_widget.options:
            load_model_widget.index = 0
    except:
        load_model_widget.options = []
load_model_refresh_button.on_click(model_list)
model_list(None)

def change_file(change):
    try:
        file = load_model_widget.value
        ts = os.path.getctime(file)
        d = datetime.datetime.fromtimestamp(ts)
        s = d.strftime('%Y-%m-%d %H:%M:%S')
        load_model_time_widget.value = s
    except:
        load_model_time_widget.value = ""
load_model_widget.observe(change_file, names='value')

def update_image(change):
    global load_flag, no
    load_flag = True
    no = no_widget.value
    load_img(no)
update_image_button.on_click(update_image)


In [ ]:
import numpy as np
from functools import partial

WIDTH = 80
HEIGHT = 80
SIZE = 8

check_image_button = ipywidgets.Button(description=f'{SIZE}個単位チェック')
check_next_images_button = ipywidgets.Button(description=f'[{SIZE}]>')
check_prev_images_button = ipywidgets.Button(description=f'<[-{SIZE}]')
check_start_index_widget = ipywidgets.IntText(description='開始位置')
check_end_index_widget = ipywidgets.IntText(description='終了位置')
check_image_count_widget = ipywidgets.IntText(description='最終画像位置')
check_update_button = ipywidgets.Button(description='更新')

# 画像を表示するウィジェット
snapshot_widgets = []
snapshot_button_widgets = []


def edit_image(index, b):
    global load_flag,no,check_flag
    no = check_no + index
    load_flag = True
    check_flag = False
    load_img(no)
    no_widget.value = no
    
for i in range(SIZE):
    image = ipywidgets.Image(width=WIDTH, height=HEIGHT)
    edit_button = ipywidgets.Button(description="編集", layout=ipywidgets.Layout(width=f'{WIDTH}px', height=f'30px'))
    edit_button.on_click(partial(edit_image, i))
    black_image = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
    image.value = bgr8_to_jpeg(black_image)
    snapshot_widgets.append(image)
    snapshot_button_widgets.append(VBox([edit_button,image]))

def get_load_dataset_length():
    xy_path = os.path.join(asset_root(), load_task_widget.value, load_datasets_widget.value, "xy")
    xy_filenames = get_file_names(xy_path)
    check_image_count_widget.value = len(xy_filenames)
    last_no = len(xy_filenames)
    return last_no

def next_images(c):
    global check_no,last_no,check_flag
    check_next_images_button.disabled = True
    check_prev_images_button.disabled = True
    write_log(f"check_flag:"+str(check_flag))
    if check_flag == False:
        check_no = no_widget.value
        check_flag = True
    else:
        check_no += SIZE
        write_log(f"check_no: {check_no}")
    try:
        last_no = get_load_dataset_length()
        if check_no < last_no:
            load_images(check_no)
        else:
            check_next_images_button.disabled = False
            check_prev_images_button.disabled = False
    except:
        check_next_images_button.disabled = False
        check_prev_images_button.disabled = False
    
def prev_images(c):
    global check_no,check_flag
    check_next_images_button.disabled = True
    check_prev_images_button.disabled = True
    write_log(f"check_flag:"+str(check_flag))
    if check_flag == False:
        check_no = no_widget.value
        check_no -= SIZE
        check_flag = True
    else:
        check_no -= SIZE
        write_log(f"check_no: {check_no}")
    try:
        last_no = get_load_dataset_length()
        if check_no < 0:
            check_no = 0
        load_images(check_no)
    except:
        check_next_images_button.disabled = False
        check_prev_images_button.disabled = False
    
def load_images(c):
    global check_no, last_no, snapshot_widgets
    write_log("画像を" + str(SIZE) + "枚読込み、推論結果を付与します。")
    try:
        xy_path = os.path.join(asset_root(), load_task_widget.value, load_datasets_widget.value, "xy")
        xy_filenames = get_file_names(xy_path)
        last_no = len(xy_filenames)
        check_image_count_widget.value = last_no
        now_no = 0
        write_log(f"{xy_path}のデータセットを読み込みます。データ数(xy): {last_no}")
        for i in range(SIZE):
            now_no = check_no + i
            if now_no < last_no:
                try:
                    xy_name = xy_filenames[now_no]
                    img = cv2.imread(xy_name)
                    preprocessed = preprocess(img)
                    output = model(preprocessed).detach().cpu().numpy().flatten()
                    result_x = output[0]
                    result_y = output[1]
                    result_speed = output[3]
                    result_x = int(IMG_WIDTH * (result_x / 2.0 + 0.5))
                    result_y = int(IMG_HEIGHT * (result_y / 2.0 + 0.5))
                    result_speed = int(IMG_HEIGHT * (result_speed / 2.0 + 0.5))
                    marked_img = cv2.circle(img, (int(result_x), int(result_y)), 8, (255, 0, 0), 3)
                    marked_img = cv2.line(marked_img,(219,224-result_speed),(219,224),(0,140,255),3)
                    marked_img = cv2.putText(marked_img,"speed:"+str(result_speed),(160,215),cv2.FONT_HERSHEY_SIMPLEX,0.3,(255,255,255))

                    snapshot_widgets[i].value = bgr8_to_jpeg(marked_img)
                    
                    time.sleep(10/1000)
                except Exception as e:
                    write_log(f"{e}")
                    black_image = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
                    snapshot_widgets[i].value = bgr8_to_jpeg(black_image)
            else:
                black_image = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
                snapshot_widgets[i].value = bgr8_to_jpeg(black_image)
        check_next_images_button.disabled = False
        check_prev_images_button.disabled = False
    except Exception as e:
        write_log(f"{e}")
    #get_jetson_nano_memory_usage()

def update_images(c):
    global check_no
    check_no = check_start_index_widget.value
    load_images(check_no)
    
check_no = 0
check_image_button.on_click(load_images)
check_prev_images_button.on_click(prev_images)
check_next_images_button.on_click(next_images)
check_update_button.on_click(update_images)

In [ ]:
movie_button = ipywidgets.Button(description='動画の作成')
movie_name_widget = ipywidgets.Text(description='動画名',value="run_video")

def make_movie(change):
    global model
    
    if not movie_name_widget.value.strip():
        write_log("ファイル名を指定してください。")
        return 
    write_log("動画を作成します。")
    path = os.path.join(asset_root(), "video")
    os.makedirs(path, exist_ok=True)
    output = os.path.join(path, movie_name_widget.value + ".mp4")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = int(30 / skip_movie_dropdown.value)
    outfh = cv2.VideoWriter(output, fourcc, fps, (224, 224))

    file_list = sorted(
        glob.glob(os.path.join(asset_root(), load_task_widget.value, load_datasets_widget.value, "xy", "*.jpg")),
        key=lambda f: extract_numbers(os.path.basename(f)),
    )

    try:
        res_num = len(file_list)
        
        count = 0
        skip_movie = skip_movie_dropdown.value
        terminal_time = 1/(30/skip_movie)
        current_time = 0
        process_time = 0
        total_process_time = 0
        for i, file_name in enumerate(file_list):
            
            if i % skip_movie == 0:
                current_time += terminal_time
                img = cv2.imread(file_name)
                
                process_time = time.time()
                preprocessed = preprocess(img)
                output = model(preprocessed).detach().cpu().numpy().flatten()
                result_x = float(output[0])
                result_y = float(output[1])
                result_x = int(IMG_WIDTH * (result_x / 2.0 + 0.5))
                result_y = int(IMG_HEIGHT * (result_y / 2.0 + 0.5))    
                img = cv2.circle(img, (int(result_x), int(result_y)), 8, (255, 0, 0), 3)

                # Speed
                result_speed = output[3]
                result_speed = int(IMG_WIDTH * (result_speed / 2.0 + 0.5))
                if result_speed > 224:
                    result_speed = 244
                elif result_speed < 0:
                    result_speed = 0
                img = cv2.line(img,(218,0),(218,224),(0,0,0),5)
                img = cv2.line(img,(219,224-result_speed),(219,224),(0,140,255),3)
                img = cv2.putText(img,"speed:"+str(result_speed),(160,215),cv2.FONT_HERSHEY_SIMPLEX,0.3,(255,255,255))
                total_process_time += time.time() - process_time 
                
                if i % (skip_movie*10) == 0:
                    write_log(f"{current_time:.1f}秒まで完了, 推論処理平均: {total_process_time/10*1000:.1f}ms, {int(i/skip_movie)}枚目/{int(res_num/skip_movie)}枚中を処理中")
                    total_process_time = 0
                outfh.write(img)
                del img
    finally:
        # エラーが発生しても確実にリソースを解放する
        outfh.release()
        write_log("動画の出力が完了しました。")
        get_jetson_nano_memory_usage()

movie_button.on_click(make_movie)

In [ ]:
import subprocess
import re

used_memory_widget = ipywidgets.IntText(description='Useメモリ', value=1)
total_memory_widget = ipywidgets.IntText(description='全メモリ', value=1)
memory_button = ipywidgets.Button(description='使用メモリ量の取得')

def get_jetson_nano_memory_usage(event=None):
    command = 'tegrastats'
    try:
        process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
        
        mem_usage_pattern = re.compile(r'RAM (\d+)/(\d+)MB')
        
        max_lines_to_read = 10
        for _ in range(max_lines_to_read):
            line = process.stdout.readline()
            if not line:
                break 
            matches = mem_usage_pattern.search(line)
            if matches:
                used_memory_widget.value = int(matches.group(1))
                total_memory_widget.value = int(matches.group(2))
                write_log("使用メモリ： " + str(used_memory_widget.value) + "/" + str(total_memory_widget.value))
                process.kill()
                return
        
        process.kill()  
        return

    except subprocess.CalledProcessError as e:
        return

get_jetson_nano_memory_usage()
memory_button.on_click(get_jetson_nano_memory_usage)

In [ ]:
separator = ipywidgets.HTML('<hr style="border-color:gray;margin:10px 0"/>')
title1 = ipywidgets.HTML('<b>【1.使用する推論モデル】</b> [New]は新規モデル。')
title2 = ipywidgets.HTML('<b>【2.読込元データセット】</b> アノテーションを実施するデータセットを選択。')
title3 = ipywidgets.HTML('<b>【3.保存先データセット】</b> データセットの保存先を選択。')
title4 = ipywidgets.HTML('<b>【4.アノテーションの実施】</b> 緑◯がアノテーション, 青◯がAIでの推論。車両の走らせたい場所で、画面をクリックすると保存先データセットのxyにデータが登録されます。Speedは[速度追加]で追加します。')
title5 = ipywidgets.HTML('<b>【5.学習】</b> EPOCH指定で学習できます。')
title6 = ipywidgets.HTML('<b>【6.評価動画の作成】</b> 動画を作成します。')

data_collection_widget = ipywidgets.VBox([
    separator,
    title1,
    ipywidgets.HBox([load_model_widget, load_model_refresh_button, load_model_time_widget, load_model_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title2,
    ipywidgets.HBox([load_datasets_widget, load_datasets_refresh_button, load_task_widget, load_image_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title3,
    ipywidgets.HBox([save_datasets_widget, save_datasets_refresh_button, save_task_widget]),
    ipywidgets.HBox([datasets_xy_count_widget,datasets_speed_count_widget]),
    ipywidgets.HBox([Label('datasetの新規作成'),datasets_name_widget,dataset_create_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title4,
    ipywidgets.HBox([no_widget,update_image_button,delete_image_button]), 
    ipywidgets.HBox([skip_dropdown,sleep_dropdown]),
    ipywidgets.HBox([picture_widget,ipywidgets.VBox([speed_slider,add_speed_button]),ipywidgets.VBox([play_button,prev_image_button,Label(f'{SIZE}個単位での処理'),check_prev_images_button]),ipywidgets.VBox([stop_button,next_image_button,Label(f''),check_next_images_button])]),
    ipywidgets.HBox(snapshot_button_widgets),
    ipywidgets.HBox([save_datasets_widget, save_datasets_refresh_button, save_task_widget]),
    ipywidgets.HBox([datasets_xy_count_widget,datasets_speed_count_widget]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title5,
    ipywidgets.HBox([save_datasets_widget, save_datasets_refresh_button, save_task_widget]),
    ipywidgets.HBox([datasets_xy_count_widget,datasets_speed_count_widget]),
    ipywidgets.HBox([epochs_widget,train_button,eval_button]),
    ipywidgets.HBox([progress_widget,loss_widget]),
    ipywidgets.HBox([save_model_name_widget, save_model_button, save_best_model_checkbox]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title6,
    ipywidgets.HBox([load_datasets_widget, load_datasets_refresh_button, load_task_widget]),
    ipywidgets.HBox([movie_name_widget,skip_movie_dropdown,movie_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
])
display(data_collection_widget)